In [6]:
import pandas as pd
import numpy as np
from io import StringIO
import os

# Affichage complet Pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)

# ==========================================================
# LOAD DATA FILE
# ==========================================================
def load_data(filename):
    path = os.path.join("..", "data", filename)
    with open(path, "r", encoding="utf-8") as f:
        return f.readlines()


# ==========================================================
# PROCESS ONE FILE
# ==========================================================
def process_file(filename):
    lines = load_data(filename)

    # ---------- FIND BLOCKS ----------
    idx_push = None
    idx_move = None
    for i, line in enumerate(lines):
        if "K-Push" in line and idx_push is None:
            idx_push = i
        if "K-Move" in line and idx_move is None:
            idx_move = i

    if idx_push is None:
        print("❌ K-PUSH NOT FOUND")
        return None, None, None, None
    if idx_move is None:
        print("❌ K-MOVE NOT FOUND")
        return None, None, None, None

    # ---------- SPLIT BLOCKS (ORDER INDEPENDENT) ----------
    if idx_push < idx_move:
        push_lines = lines[idx_push:idx_move]
        move_lines = lines[idx_move:]
    else:
        move_lines = lines[idx_move:idx_push]
        push_lines = lines[idx_push:]

    # ---------- CLEAN K-PUSH ----------
    try:
        header_idx_push = next(i for i, line in enumerate(push_lines) if "temps" in line.lower())
    except StopIteration:
        print("❌ Header K-Push non trouvé")
        return None, None, None, None

    push_str = "".join(push_lines[header_idx_push:])
    df_push = pd.read_csv(StringIO(push_str), sep=r"\t|,", engine="python", on_bad_lines="skip")
    df_push = df_push.dropna(axis=1, how='all')
    df_push = df_push.rename(columns={"temps (seconde)": "time", "CHANNEL_1": "force"})
    if "time" in df_push.columns and "force" in df_push.columns:
        df_push = df_push[["time", "force"]]
        df_push = df_push.apply(pd.to_numeric, errors='coerce').dropna()
    else:
        df_push = pd.DataFrame()

    # ---------- BASELINE ----------
    baseline = {"wrist": [], "shoulder": []}
    for line in lines:
        if "Quaternion de base" in line:
            parts = line.strip().split("\t")
            if len(parts) < 5:
                parts = line.strip().split(",")
            values = []
            for x in parts[1:]:
                try:
                    values.append(float(x))
                except:
                    continue
            values = values[:4]
            if "S121577" in parts[0]:
                baseline["wrist"] = values
            elif "S121578" in parts[0]:
                baseline["shoulder"] = values

    # ---------- CLEAN K-MOVE ----------
    try:
        header_idx_move = next(i for i, line in enumerate(move_lines) if "temps" in line.lower())
    except StopIteration:
        print("❌ Header K-Move non trouvé")
        return df_push, pd.DataFrame(), pd.DataFrame(), baseline

    move_data_lines = move_lines[header_idx_move + 1:]
    split_rows = []
    for line in move_data_lines:
        row = line.strip().split("\t")
        if len(row) < 2:
            row = line.strip().split(",")
        split_rows.append(row)

    df_move = pd.DataFrame(split_rows)
    df_move = df_move.replace("", np.nan).dropna(axis=1, how='all')
    df_move = df_move.apply(pd.to_numeric, errors='coerce').ffill().bfill()

    # ---------- EXTRACT SENSORS ----------
    try:
        df_wrist = df_move.iloc[:, [0,1,2,3,4]].copy()
        df_wrist.columns = ["time","qx","qy","qz","qw"]
        df_shoulder = df_move.iloc[:, [0,5,6,7,8]].copy()
        df_shoulder.columns = ["time","qx","qy","qz","qw"]
    except:
        df_wrist = pd.DataFrame()
        df_shoulder = pd.DataFrame()

    # ---------- RETURN ----------
    return df_push, df_wrist, df_shoulder, baseline


# ==========================================================
# MAIN SCRIPT
# ==========================================================
files = ["Data_Ch_D.csv"]  # liste des fichiers

results = {}

for file in files:
    df_push, df_wrist, df_shoulder, baseline = process_file(file)
    results[file] = {"push": df_push, "wrist": df_wrist, "shoulder": df_shoulder, "baseline": baseline}

    # ---------- FULL DISPLAY ----------
    print(f"\n===== FILE: {file} =====")
    print("\nK-Push:\n", df_push.head())
    print("\nWrist (S121577):\n", df_wrist.head())
    print("\nShoulder (S121578):\n", df_shoulder.head())
    print("\nBaseline wrist:", baseline.get("wrist", []))
    print("Baseline shoulder:", baseline.get("shoulder", []))
    print("\nShapes:", df_push.shape, df_wrist.shape, df_shoulder.shape)


===== FILE: Data_Ch_D.csv =====

K-Push:
     time  force
0  0.000    0.0
1  0.001    0.0
2  0.002    0.0
3  0.003    0.0
4  0.004    0.0

Wrist (S121577):
     time        qx        qy        qz        qw
0  0.000  0.396362  0.516846  0.433289  0.622803
1  0.004  0.396376  0.516894  0.433273  0.622825
2  0.008  0.396373  0.516982  0.433239  0.622789
3  0.012  0.396370  0.517070  0.433206  0.622754
4  0.016  0.396342  0.517134  0.433178  0.622728

Shoulder (S121578):
     time        qx        qy        qz        qw
0  0.000 -0.573059  0.038269  0.100220  0.812439
1  0.004 -0.573072  0.038239  0.100222  0.812427
2  0.008 -0.573105  0.038179  0.100222  0.812400
3  0.012 -0.573163  0.038148  0.100252  0.812365
4  0.016 -0.573195  0.038117  0.100283  0.812336

Baseline wrist: [0.383, 0.513, 0.399, 0.656]
Baseline shoulder: [-0.568, 0.031, 0.1, 0.816]

Shapes: (47382, 2) (11835, 5) (11835, 5)


In [9]:
## STEP 2 - CONVERT QUATERNION TO ANATOMICAL ANGLE + PLOT

import numpy as np
import matplotlib.pyplot as plt

# ==========================================================
# FUNCTION: CONVERT QUATERNION TO ANATOMICAL ANGLE
# ==========================================================
def quaternion_to_angle_x_corrected(df_quat, baseline_deg):
    """
    Convert quaternion to anatomically meaningful wrist/elbow angle.

    Steps:
    1. Compute raw angle from quaternion
    2. Unwrap angle to avoid discontinuities (-180/+180 jumps)
    3. Convert to degrees
    4. Recalibrate so that:
       - max flexion = baseline_deg (manual reference from video)
       - extension increases toward ~180°
    """

    # Extract quaternion components
    qx = df_quat["qx"].values
    qy = df_quat["qy"].values
    qz = df_quat["qz"].values
    qw = df_quat["qw"].values

    # ---------- STEP 1: RAW ANGLE (rad) ----------
    angle_raw = np.arctan2(
        2*(qw*qx + qy*qz),
        1 - 2*(qx**2 + qy**2)
    )

    # ---------- STEP 2: UNWRAP ----------
    angle_unwrapped = np.unwrap(angle_raw)

    # ---------- STEP 3: CONVERT TO DEGREES ----------
    angle_deg = np.degrees(angle_unwrapped)

    # ---------- STEP 4: ZERO-REFERENCE ----------
    angle_zeroed = angle_deg - angle_deg[0]

    # Invert direction if needed (flexion = small, extension = large)
    if np.mean(angle_zeroed) < 0:
        angle_zeroed = -angle_zeroed

    # ---------- STEP 5: APPLY MANUAL ANATOMICAL SCALE ----------
    angle_anatomical = angle_zeroed + baseline_deg

    # Store in DataFrame
    df_angles = df_quat[["time"]].copy()
    df_angles["angle"] = angle_anatomical

    return df_angles


# ==========================================================
# MULTI-FILE PROCESSING
# ==========================================================
files = ["Data_Ch_D.csv"]  # Example single file
# files = ["Data_droite.csv", "Data_GG.csv"]  # Multiple files

# Manual baseline angles for each file (from video analysis)
baseline_angles = {
    "Data_Ch_D.csv": 40,
   
}

# Store results
angles_results = {}

for file in files:
    # Get wrist DataFrame from previous step (process_file)
    df_wrist = results[file]["wrist"]
   

    # Get manual baseline angle for this file
    manual_baseline = baseline_angles[file]

    # Convert quaternion → anatomical angle
    df_wrist_angles = quaternion_to_angle_x_corrected(df_wrist, manual_baseline)

    # Store results
    angles_results[file] = df_wrist_angles


    df_angles = quaternion_to_angle_x_corrected(df_wrist, baseline_deg=0)

# Lissage et resample
df_angles = df_angles.set_index("time")
df_angles["angle"] = df_angles["angle"].rolling(5, center=True).mean()  # lissage
df_angles = df_angles.resample("8ms").mean().interpolate()
df_angles = df_angles.reset_index()

# Normalisation sur -9 → +9
angle_min = df_angles["angle"].min()
angle_max = df_angles["angle"].max()
df_angles["angle_physio"] = (df_angles["angle"] - angle_min) / (angle_max - angle_min) * 18 - 9

# Plot
plt.figure(figsize=(12,5))
plt.plot(df_angles["time"], df_angles["angle_physio"], label="Wrist Angle")
plt.xlabel("Time (s)")
plt.ylabel("Angle (°)")
plt.title("Wrist Angle (X) - Physiological Range")
plt.grid(True)
plt.legend()
plt.show()

    # ---------- PLOT ----------
    plt.figure(figsize=(12,5))
    plt.plot(df_wrist_angles["time"], df_wrist_angles["angle"], label=f"Wrist Angle ({file})")
    plt.xlabel("Time (s)")
    plt.ylabel("Angle (degrees)")
    plt.title(f"Wrist Angle Over Time (Corrected & Anatomical) - {file}")
    plt.legend()
    plt.grid(True)
    plt.show()


    # ---------- CHECK RANGE ----------
    print(f"===== FILE: {file} =====")
    print("Min angle:", df_wrist_angles["angle"].min())
    print("Max angle:", df_wrist_angles["angle"].max())

IndentationError: unexpected indent (453344290.py, line 111)

In [10]:
## STEP 2 - CONVERT QUATERNION TO ANATOMICAL ANGLE + PLOT

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================================
# FUNCTION: CONVERT QUATERNION TO ANATOMICAL ANGLE
# ==========================================================
def quaternion_to_angle_x_corrected(df_quat, baseline_deg=0, smooth_window=5, resample_ms=8):
    """
    Convert quaternion to anatomically meaningful wrist/elbow angle with:
    - unwrapping
    - smoothing (rolling mean)
    - resampling (to avoid staircase effect)
    - normalization to physiological range (~±9°)

    Returns a DataFrame with columns: time, angle_raw, angle_physio
    """

    qx = df_quat["qx"].values
    qy = df_quat["qy"].values
    qz = df_quat["qz"].values
    qw = df_quat["qw"].values

    # ---------- STEP 1: RAW ANGLE ----------
    angle_raw = np.arctan2(
        2*(qw*qx + qy*qz),
        1 - 2*(qx**2 + qy**2)
    )

    # ---------- STEP 2: UNWRAP ----------
    angle_unwrapped = np.unwrap(angle_raw)

    # ---------- STEP 3: CONVERT TO DEGREES ----------
    angle_deg = np.degrees(angle_unwrapped)

    # ---------- STEP 4: ZERO-REFERENCE ----------
    angle_zeroed = angle_deg - angle_deg[0]
    if np.mean(angle_zeroed) < 0:
        angle_zeroed = -angle_zeroed

    # ---------- STEP 5: APPLY MANUAL BASELINE ----------
    angle_anatomical = angle_zeroed + baseline_deg

    # ---------- STEP 6: CREATE DATAFRAME ----------
    df_angles = df_quat[["time"]].copy()
    df_angles["angle_raw"] = angle_anatomical

    # ---------- STEP 7: SMOOTH ----------
    df_angles = df_angles.set_index("time")
    df_angles["angle_smooth"] = df_angles["angle_raw"].rolling(smooth_window, center=True).mean()

    # ---------- STEP 8: RESAMPLE ----------
    df_angles = df_angles.resample(f"{resample_ms}ms").mean().interpolate()

    # ---------- STEP 9: NORMALIZE TO PHYSIO RANGE (-9° → +9°) ----------
    min_angle = df_angles["angle_smooth"].min()
    max_angle = df_angles["angle_smooth"].max()
    df_angles["angle_physio"] = (df_angles["angle_smooth"] - min_angle) / (max_angle - min_angle) * 18 - 9

    df_angles = df_angles.reset_index()
    return df_angles


# ==========================================================
# MULTI-FILE PROCESSING
# ==========================================================
files = ["Data_Ch_D.csv"]  # liste des fichiers
baseline_angles = {"Data_Ch_D.csv": 40}  # angle de référence manuel

angles_results = {}

for file in files:
    # Récupère les données du poignet depuis STEP 1
    df_wrist = results[file]["wrist"]

    # Récupère l'angle baseline pour ce fichier
    manual_baseline = baseline_angles[file]

    # Convert quaternion → angle anatomique + lissage + resample + normalization
    df_angles = quaternion_to_angle_x_corrected(df_wrist, baseline_deg=manual_baseline)

    # Stocke les résultats
    angles_results[file] = df_angles

    # ---------- PLOT ----------
    plt.figure(figsize=(12,5))
    plt.plot(df_angles["time"], df_angles["angle_physio"], label=f"Wrist Angle ({file})")
    plt.xlabel("Time (s)")
    plt.ylabel("Angle (°)")
    plt.title(f"Wrist Angle X - Physiological Range - {file}")
    plt.grid(True)
    plt.legend()
    plt.show()

    # ---------- CHECK RANGE ----------
    print(f"===== FILE: {file} =====")
    print("Min angle (physio):", df_angles["angle_physio"].min())
    print("Max angle (physio):", df_angles["angle_physio"].max())

TypeError: Only valid with DatetimeIndex, TimedeltaIndex or PeriodIndex, but got an instance of 'Index'

In [13]:

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# ==========================================================
# FUNCTION: CONVERT QUATERNION TO ANATOMICAL ANGLE
# ==========================================================
def quaternion_to_angle_x_corrected(df_quat, baseline_deg=0, smooth_window=5, resample_ms=8):
    """
    Convert quaternion to anatomically meaningful wrist/elbow angle with:
    - unwrapping
    - smoothing (rolling mean)
    - resampling (to avoid staircase effect)
    - normalization to physiological range (~±9°)
    """

    qx = df_quat["qx"].values
    qy = df_quat["qy"].values
    qz = df_quat["qz"].values
    qw = df_quat["qw"].values

    # ---------- STEP 1: RAW ANGLE ----------
    angle_raw = np.arctan2(
        2*(qw*qx + qy*qz),
        1 - 2*(qx**2 + qy**2)
    )

    # ---------- STEP 2: UNWRAP ----------
    angle_unwrapped = np.unwrap(angle_raw)

    # ---------- STEP 3: CONVERT TO DEGREES ----------
    angle_deg = np.degrees(angle_unwrapped)

    # ---------- STEP 4: ZERO-REFERENCE ----------
    angle_zeroed = angle_deg - angle_deg[0]
    if np.mean(angle_zeroed) < 0:
        angle_zeroed = -angle_zeroed

    # ---------- STEP 5: APPLY MANUAL BASELINE ----------
    angle_anatomical = angle_zeroed + baseline_deg

    # ---------- STEP 6: CREATE DATAFRAME ----------
    df_angles = df_quat[["time"]].copy()
    df_angles["angle_raw"] = angle_anatomical

    # ---------- STEP 7: SMOOTH ----------
    df_angles["angle_smooth"] = df_angles["angle_raw"].rolling(smooth_window, center=True).mean()

    # ---------- STEP 8: RESAMPLE ----------
    # Convert time (s) to TimedeltaIndex
    df_angles.index = pd.to_timedelta(df_angles["time"], unit='s')
    df_angles = df_angles.resample(f"{resample_ms}ms").mean().interpolate()
    df_angles = df_angles.reset_index()
    df_angles.rename(columns={"index":"time"}, inplace=True)

    # ---------- STEP 9: NORMALIZE TO PHYSIO RANGE (-9° → +9°) ----------
    min_angle = df_angles["angle_smooth"].min()
    max_angle = df_angles["angle_smooth"].max()
    df_angles["angle_physio"] = (df_angles["angle_smooth"] - min_angle) / (max_angle - min_angle) * 18 - 9

    return df_angles

# ==========================================================
# MULTI-FILE PROCESSING
# ==========================================================
files = ["Data_Ch_D.csv"]  # Liste de fichiers
baseline_angles = {"Data_Ch_D.csv": 40}  # Baseline manuel pour chaque fichier
angles_results = {}

for file in files:
    # Récupère le DataFrame wrist depuis la STEP 1
    df_wrist = results[file]["wrist"]

    # Baseline manuel pour ce fichier
    manual_baseline = baseline_angles[file]

    # ---------- CONVERT QUATERNION → ANGLE ANATOMIQUE ----------
    df_angles = quaternion_to_angle_x_corrected(df_wrist, baseline_deg=manual_baseline)

    # Stocke les résultats
    angles_results[file] = df_angles

    # ---------- PLOT ----------
    plt.figure(figsize=(12,5))
    plt.plot(df_angles["time"].dt.total_seconds(), df_angles["angle_physio"], label=f"Wrist Angle ({file})")
    plt.xlabel("Time (s)")
    plt.ylabel("Angle (°)")
    plt.title(f"Wrist Angle (X) - Physiological Range - {file}")
    plt.grid(True)
    plt.legend()
    plt.show()

    # ---------- CHECK RANGE ----------
    print(f"===== FILE: {file} =====")
    print("Min angle:", df_angles["angle_physio"].min())
    print("Max angle:", df_angles["angle_physio"].max())

ValueError: cannot insert time, already exists